# STAC data prep — Soil depth to bedrock (single large COG)

Notebook to migrate the SoilGrids250m absolute depth to bedrock raster to one **Cloud Optimized GeoTIFF (CoG)**, published in GeoServer WMS.

- **Case:** one global 2D raster, 172800 x 67200 pixels (250 m, about 0.0021 degree), `int32`, depth in cm (0-5000 and above). The file is 8.5 GB compressed and **46 GB uncompressed**, so it is never read as a whole: GDAL streams it to the COG block by block.
- **Format:** one COG with 512 x 512 internal tiles and overviews (see `DATA_FORMATS.md`). The overviews make a WMS request for the whole world fast.
- **Resampling:** `AVERAGE` for overviews (continuous quantity); GDAL excludes nodata from the average.
- **Values:** stored values and the `-32768` nodata are kept (no cast), so the COG is lossless.
- **Metadata template:** `metadata/metadata_Soil_Depth.json` (Hengl et al. 2017, ISRIC). The empty fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\Soil_depth.tif`.

Runtime: creating the COG reads the whole raster once over the network drive and recompresses 46 GB; expect on the order of an hour and about 10 GB of free space in the temp folder (the output is first written to a local temp folder and then copied to `P:`).

**Running in docker:** the `P:` drive is not visible in the container, so run from the host:

```powershell
docker\run_notebook_with_p_drive.ps1 -Notebook 26_soil_depth.ipynb -Variable soil_depth -InputPattern "Soil_depth.tif"
```

### Import packages

In [1]:
import json
import os
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import rasterio
from rasterio.shutil import copy as rio_copy
from rasterio.windows import Window

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
COLLECTION_ID = "soil_depth"
ITEM_ID = "soil_depth"
OUT_NODATA = -32768  # same as the source, kept so that the COG is lossless
UNITS = "cm"
TEMPORAL_EXTENT = ["1950-01-01T00:00:00Z", "2015-12-31T23:59:59Z"]  # from the metadata template (1950-2015)

data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_tif = data_vivian / "Soil_depth.tif"
metadata_template_path = stac_repo / "metadata" / "metadata_Soil_Depth.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
cog_dir = processed_data_dir / "cog"
cog_dir.mkdir(parents=True, exist_ok=True)
cog_path = cog_dir / f"{ITEM_ID}.tif"

for p in (source_tif, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_tif)
print("Metadata template:", metadata_template_path)
print("Output COG:", cog_path)

Source: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\Soil_depth.tif
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_Soil_Depth.json
Output COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_depth\cog\soil_depth.tif


### Read raw data

Only the header and a few windows are read here; the raster is too big to load.

In [3]:
with rasterio.open(source_tif) as src:
    print(f"{src.width} x {src.height} | dtype {src.dtypes[0]} | nodata {src.nodata} | crs {src.crs.to_epsg()}")
    print(f"bounds {tuple(round(v, 5) for v in src.bounds)} | res {tuple(round(v, 7) for v in src.res)}")
    print(f"tiled {src.is_tiled} | compression {src.compression} | overviews {src.overviews(1)}")
    print("tags:", src.tags(1))
    print(f"uncompressed size: {src.width * src.height * 4 / 1e9:.1f} GB")
    src_profile = src.profile
    src_bounds = src.bounds
    src_res = src.res
    assert src.crs.to_epsg() == 4326
    assert src.nodata == OUT_NODATA

    # Value range from the coarsest overview (a few thousand pixels), enough as a sanity check
    ovr = src.read(1, out_shape=(src.height // 128, src.width // 128))
    valid = ovr[ovr != src.nodata]
    print(f"coarse preview: valid {valid.size} of {ovr.size} | min {valid.min()} mean {valid.mean():.0f} max {valid.max()}")

172800 x 67200 | dtype int32 | nodata -32768.0 | crs 4326
bounds (-180.0, -56.00081, 179.99994, 83.99917) | res (0.0020833, 0.0020833)
tiled False | compression Compression.deflate | overviews [2, 4, 8, 16, 32, 64, 128]
tags: {}
uncompressed size: 46.4 GB
coarse preview: valid 200283 of 708750 | min 0 mean 3926 max 59024


### Check CF compliancy original file

Not applicable for a plain GeoTIFF: CRS, nodata and attributes are written explicitly in the metadata file.

### Make CF compliant alterations (dataset dependent)

- Keep the source values and the `-32768` nodata (no cast, so the COG is lossless).
- Fill the metadata fields that are empty in the template using the raster: `SPATIAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `DATA_MODEL`, `COLLECTION_ID`, `UNITS`, ...
- `ITEMS` holds the item title and description for the STAC builder.

In [4]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

left, bottom, right, top = src_bounds
bbox = [max(left, -180.0), max(bottom, -90.0), min(right, 180.0), min(top, 90.0)]

metadata.update({
    "TITLE_ABBREVIATION": "SoilDepth",
    "SHORT_DESCRIPTION": "Absolute depth to bedrock in cm at 250 m resolution, globally.",
    "INSTITUTION": "ISRIC - World Soil Information",
    "HISTORY": [
        "SoilGrids250m absolute depth to bedrock (BDTICM), Hengl et al. (2017), distributed by ISRIC",
        "Converted to a Cloud Optimized GeoTIFF (512 x 512 tiles, DEFLATE with horizontal predictor, average overviews); values and nodata unchanged",
    ],
    "MEDIA_TYPE": "image/tiff; application=geotiff; profile=cloud-optimized",
    "DATA_MODEL": "raster_cog",
    "DIMENSIONS": ["x", "y"],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    "TEMPORAL_EXTENT": TEMPORAL_EXTENT,
    "KEYWORDS": ["depth to bedrock", "soil", "SoilGrids", "bedrock", "global"],
    "TAGS": ["soil", "depth", "bedrock", "cog"],
    "LONG_NAME": "Absolute depth to bedrock",
    "UNITS": UNITS,
    "COMMENT": (
        "Absolute depth to bedrock in cm, predicted from a global compilation of soil observations (Hengl et al. 2017). "
        f"{OUT_NODATA} is nodata (no prediction). The file is a static map, not a time series; the 1950-2015 period "
        "is the period of the underlying observations."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(abs(src_res[0]), 7),
    "NODATA": OUT_NODATA,
    "DATA_TYPE": src_profile["dtype"],
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": COLLECTION_ID,  # GeoServer workspace used by scripts/26_soil_depth.ipynb
    "ITEMS": {
        ITEM_ID: {
            "TITLE": "Absolute depth to bedrock (250 m)",
            "DESCRIPTION": "SoilGrids250m absolute depth to bedrock in cm, global, 250 m resolution.",
        }
    },
})
metadata["PROVIDERS"]["name"] = metadata["PROVIDERS"]["name"] or "ISRIC - World Soil Information"

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("NODATA:", metadata["NODATA"], "| DATA_TYPE:", metadata["DATA_TYPE"])

SPATIAL_EXTENT: [-180.0, -56.00081, 179.999942, 83.999167]
SPATIAL_RESOLUTION: 0.0020833
NODATA: -32768 | DATA_TYPE: int32


### Write data to CoG

GDAL's `COG` driver reads the source block by block, so the 46 GB are never in memory. The COG is written to a local temp folder and then copied to `P:` (writing the intermediate tiled file over the network drive is much slower).

In [ ]:
t0 = time.time()
with tempfile.TemporaryDirectory(prefix="soil_depth_cog_") as tmp:
    tmp_cog = Path(tmp) / cog_path.name
    rio_copy(
        source_tif, tmp_cog,
        driver="COG",
        COMPRESS="DEFLATE",
        PREDICTOR="2",  # horizontal differencing, good for integers
        BLOCKSIZE=512,
        OVERVIEW_RESAMPLING="AVERAGE",
        BIGTIFF="IF_SAFER",
        NUM_THREADS="ALL_CPUS",
    )
    print(f"COG built in {time.time() - t0:.0f} s ({tmp_cog.stat().st_size / 1e6:.1f} MB)")
    shutil.copyfile(tmp_cog, cog_path)
print("Wrote COG:", cog_path, f"({cog_path.stat().st_size / 1e6:.1f} MB)")

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print("Wrote metadata:", metadata_output_path)

### Verify the output

Structure is checked on the header; values are compared at full resolution on several windows (a full comparison would read 46 GB twice).

In [ ]:
# Windows in lon/lat: Nile delta, Mekong delta, Netherlands, Himalaya
WINDOWS = {
    "nile_delta": (29.0, 33.0, 29.0, 32.0),
    "mekong_delta": (104.0, 108.0, 8.0, 12.0),
    "netherlands": (3.0, 7.0, 50.0, 54.0),
    "himalaya": (85.0, 89.0, 26.0, 30.0),
}

with rasterio.open(cog_path) as dst, rasterio.open(source_tif) as src:
    assert dst.dtypes[0] == src.dtypes[0]
    assert dst.nodata == OUT_NODATA
    assert dst.crs.to_epsg() == 4326
    assert dst.shape == src.shape and dst.transform == src.transform
    assert dst.is_tiled, "COG must be internally tiled"
    assert len(dst.overviews(1)) > 0, "COG must have overviews"
    print(f"block {dst.profile.get('blockxsize')} x {dst.profile.get('blockysize')} | overviews {dst.overviews(1)} | compression {dst.compression}")

    for name, (lon0, lon1, lat0, lat1) in WINDOWS.items():
        row0, col0 = src.index(lon0, lat1)
        row1, col1 = src.index(lon1, lat0)
        if row0 < 0 or col0 < 0 or row1 > src.height or col1 > src.width:
            print(f"{name}: outside the raster, skipped")
            continue
        win = Window(col0, row0, col1 - col0, row1 - row0)
        old, new = src.read(1, window=win), dst.read(1, window=win)
        assert np.array_equal(old, new), f"Pixel values changed in window {name}"
        print(f"{name}: {old.shape} identical | valid {(old != OUT_NODATA).mean():.0%} | max {old.max()} cm")
print("OK: soil depth COG and metadata are ready for the STAC builder notebook.")

block 512 x 512 | overviews [2, 4, 8, 16, 32, 64, 128] | compression Compression.deflate
nile_delta: (1440, 1920) identical | valid 72% | max 18260 cm
mekong_delta: (1920, 1920) identical | valid 52% | max 46917 cm
netherlands: (1920, 1920) identical | valid 63% | max 5096 cm
himalaya: (1920, 1920) identical | valid 94% | max 26405 cm
OK: soil depth COG and metadata are ready for the STAC builder notebook.
